

### Function Explanation: `calculate_activity_rate_change`


"Next we are going to overview a Python function called `calculate_activity_rate_change` to calculate this feature.
Let's go through it step-by-step.

---

### **Imports**

First, we import two key libraries:
* `import pandas as pd`: We use **pandas** because it's the industry standard for data manipulation in Python. It allows us to work with our data in a structured format called a DataFrame, which is essentially a smart spreadsheet.
* `from datetime import datetime, timedelta`: We import `datetime` and `timedelta` to handle all of our time-based calculations. This is crucial for defining our historical and current time windows accurately.

---

### **Function Definition**

Our function, `calculate_activity_rate_change`, takes two arguments: the `user_id` we want to investigate and the pandas `df`, or DataFrame, which contains all our user activity data. It's designed to return a single floating-point number, which will be the percentage change in activity.

---

### **Step 1: Data Preparation**

Inside the function, the first thing we do is prepare our data.
1.  We ensure the `timestamp` column is converted to a proper datetime format using `pd.to_datetime()`. This is critical for accurate time-based filtering.
2.  Next, we filter the main DataFrame to create a smaller one, `user_df`, that contains *only* the `api_call` activities for the *specific user* we're analyzing. This ensures we're only measuring the activity we care about for the correct person.

---

### **Step 2: Calculate the Historical Average**

Now for the core of our relative feature: establishing a baseline.
1.  We define our historical window: from seven days ago up to the current moment.
2.  We filter our user's data to get only the API calls that occurred within this 7-day window.
3.  Then, we calculate the `historical_hourly_avg`. Instead of just dividing by the number of days, we calculate the total number of hours in the period to get a more precise hourly average. This accounts for users who may have signed up only a few days ago. If there's no history, we simply set this average to zero.

---

### **Step 3: Calculate Current Activity**

Once we have our historical baseline, we need to measure the user's current activity.
1.  We define a `current_hour_df` by filtering for all API calls made by the user in just the last hour.
2.  We then simply count the number of rows in this DataFrame to get the `current_hour_calls`.

---

### **Step 4: The Final Calculation**

Finally, we compare the present to the past.
1.  We first check if the `historical_hourly_avg` is zero. If it is, and the user has made calls in the last hour, we have a potential "new account abuse" scenario. We can't divide by zero, so we return a very high number like `999.0` to signal a massive, potentially suspicious change.
2.  If there *is* a historical average, we use the standard formula for percentage change: we take the new value (`current_hour_calls`), subtract the old value (`historical_hourly_avg`), divide the result by the old value, and multiply by 100.

This gives us the final percentage. A large positive number tells us the user is suddenly far more active than usual—a key red flag for a fraud analyst.

**(End of Voiceover)**

In [ ]:
# path = "/content/drive/MyDrive/Consulting/Udemy Course/ Fraud Fundamentals Course/Scripts/user_activity.csv"

# This will be the example for the course!

In [ ]:
import pandas as pd
from google.colab import drive

# --- Step 1: Mount your Google Drive ---
# This will prompt you for authorization. Follow the link, get the authorization code,
# and paste it into the input box in your Colab notebook.
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"An error occurred while mounting Google Drive: {e}")

# --- Step 2: Define the file path and read the CSV ---
# The local path "G:\Мой диск\" corresponds to "/content/drive/My Drive/" in Colab.
# The rest of the path is the folder structure within your Google Drive.
file_path = '/content/drive/MyDrive/Consulting/Udemy Course/ Fraud Fundamentals Course/Scripts/user_activity.csv'

try:
    # Read the CSV file into a pandas DataFrame
    activity_df = pd.read_csv(file_path)

    # --- Step 3: Clean and Verify the data ---
    print(f"\nSuccessfully loaded CSV from: {file_path}")

    # Convert timestamp column, coercing errors to NaT (Not a Time)
    original_row_count = len(activity_df)
    activity_df['timestamp'] = pd.to_datetime(activity_df['timestamp'], errors='coerce')

    # Drop rows where the timestamp could not be parsed
    activity_df.dropna(subset=['timestamp'], inplace=True)
    cleaned_row_count = len(activity_df)

    rows_dropped = original_row_count - cleaned_row_count
    if rows_dropped > 0:
        print(f"Cleaned data: Removed {rows_dropped} rows with invalid timestamp formats.")

    print("\nHere are the first 5 rows of your cleaned data:")
    # Display the first 5 rows of the DataFrame
    print(activity_df.head())

except FileNotFoundError:
    print(f"\nERROR: The file was not found at the specified path: {file_path}")
    print("Please make sure the file name is correct and it is in the correct folder in your Google Drive.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# Now you can pass the 'activity_df' DataFrame to your analysis function.
# For example:
# user_to_check = 'usr_outlier_spike'
# change = calculate_activity_rate_change(user_to_check, activity_df)
# print(f"\nPercentage change for {user_to_check}: {change:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!

Successfully loaded CSV from: /content/drive/MyDrive/Consulting/Udemy Course/ Fraud Fundamentals Course/Scripts/user_activity.csv
Cleaned data: Removed 2 rows with invalid timestamp formats.

Here are the first 5 rows of your cleaned data:
            timestamp       user_id activity_type     ip_address    device_id  \
0 2025-08-04 09:05:12  usr_a8d6b2c7     page_view    89.12.34.56  dev_x_12345   
1 2025-08-04 09:05:20  usr_a8d6b2c7      api_call    89.12.34.56  dev_x_12345   
2 2025-08-04 09:06:15  usr_b3e8c1d9     page_view    192.0.2.101  dev_w_54321   
3 2025-08-04 09:07:33  usr_c4d5e6f7   transaction  198.51.100.22  dev_z_13579   
4 2025-08-04 09:08:01  usr_a8d6b2c7      api_call    89.12.34.56  dev_x_12345   

                                             details  
0                                 {"page": "/login"} 

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def calculate_activity_rate_change(user_id: str, df: pd.DataFrame) -> float:
    """
    Calculates the percentage change in API calls for a user in the current hour
    compared to their hourly average over the last 7 days.

    Args:
        user_id: The ID of the user to analyze.
        df: A pandas DataFrame with user activity data.
            Must contain 'timestamp', 'user_id', and 'activity_type' columns.

    Returns:
        The percentage change in activity. Returns 0.0 if there is no historical data.
    """
    # Ensure the timestamp column is in datetime format
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Filter for the specific user and only 'api_call' activities
    user_df = df[(df['user_id'] == user_id) & (df['activity_type'] == 'api_call')].copy()

    if user_df.empty:
        print(f"No API call data found for user: {user_id}")
        return 0.0

    # --- 1. Calculate Historical Average ---

    # Define the time window for the last 7 days
    now = datetime.now()
    seven_days_ago = now - timedelta(days=7)

    # Filter data for the last 7 days
    historical_df = user_df[(user_df['timestamp'] >= seven_days_ago) & (user_df['timestamp'] < now)]

    if historical_df.empty:
        print(f"No historical API call data in the last 7 days for user: {user_id}")
        # Handle case with no historical data to avoid division by zero
        historical_hourly_avg = 0
    else:
        # Calculate the total number of hours in the historical period for an accurate average
        total_hours_in_period = (now - historical_df['timestamp'].min()).total_seconds() / 3600
        if total_hours_in_period < 1:
            total_hours_in_period = 1 # Avoid division by zero if period is less than an hour

        # Calculate the average number of API calls per hour
        historical_hourly_avg = len(historical_df) / total_hours_in_period

    # --- 2. Calculate Current Hour's Activity ---

    # Define the window for the current hour
    one_hour_ago = now - timedelta(hours=1)

    # Filter data for the current hour
    current_hour_df = user_df[(user_df['timestamp'] >= one_hour_ago) & (user_df['timestamp'] <= now)]
    current_hour_calls = len(current_hour_df)

    # --- 3. Calculate Percentage Change ---

    if historical_hourly_avg == 0:
        if current_hour_calls > 0:
            # If there's no history but there is current activity, it's an infinite increase.
            # We can return a large number to signify a significant change.
            return 999.0
        else:
            # No history and no current activity means no change.
            return 0.0

    # Formula for percentage change: ((New - Old) / Old) * 100
    percentage_change = ((current_hour_calls - historical_hourly_avg) / historical_hourly_avg) * 100

    return percentage_change

# --- Example Usage ---

# Create a sample DataFrame for demonstration
# In a real scenario, you would load this from your user_activity.csv file
now = datetime.now()
data = {
    'timestamp': [now - timedelta(days=8)] + [now - timedelta(days=i, hours=j) for i in range(1, 8) for j in range(5)] + [now - timedelta(minutes=k) for k in range(1, 21)],
    'user_id': ['usr_a8d6b2c7'] * 36 + ['usr_outlier_spike'] * 20,
    'activity_type': ['api_call'] * 56,
    'details': ['{}'] * 56
}
sample_df = pd.DataFrame(data)

# --- Test Case 1: Normal User ---
# This user has consistent historical activity, so the change should be minimal.
normal_user_id = 'usr_a8d6b2c7'
change_normal = calculate_activity_rate_change(normal_user_id, sample_df)
print(f"Analyzing user: {normal_user_id}")
print(f"Percentage change in API calls: {change_normal:.2f}%")
print("-" * 30)

# --- Test Case 2: Outlier User with a Spike ---
# This user has no historical data but a sudden burst of 20 calls in the last hour.
# This should result in a very high percentage change.
outlier_user_id = 'usr_outlier_spike'
change_outlier = calculate_activity_rate_change(outlier_user_id, sample_df)
print(f"Analyzing user: {outlier_user_id}")
print(f"Percentage change in API calls: {change_outlier:.2f}%")

Analyzing user: usr_a8d6b2c7
Percentage change in API calls: -100.00%
------------------------------
Analyzing user: usr_outlier_spike
Percentage change in API calls: 0.00%


In [ ]:
# Now you can pass the 'activity_df' DataFrame to your analysis function.
# For example:
user_to_check = 'usr_outlier_spike'
change = calculate_activity_rate_change(user_to_check, activity_df)
print(f"\nPercentage change for {user_to_check}: {change:.2f}%")


Percentage change for usr_outlier_spike: -100.00%
